<a href="https://colab.research.google.com/github/sonjoy1s/ML/blob/main/Water_Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier


In [ ]:
df = pd.read_csv("/content/water_potability.csv")

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe().T

In [ ]:
df.columns

In [ ]:
X = df.drop('Potability',axis=1)
y = df['Potability']

In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(data=X)
plt.show()

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df['ph'].fillna(df['ph'].median(),inplace=True)
df['Sulfate'].fillna(df['Sulfate'].median(),inplace=True)
df['Trihalomethanes'].fillna(df['Trihalomethanes'].median(),inplace=True)


In [ ]:
df.isna().sum()

In [ ]:
numeric_features = X.select_dtypes(include = ['int64','float64']).columns
categorical_features = X.select_dtypes(include = ['object']).columns

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())  # NN / SVC only
])


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
# base learner
lgr = LogisticRegression(class_weight={0:1, 1:1998/1278}, max_iter=1000)
dtc = DecisionTreeClassifier()
knn = KNeighborsClassifier()
rf_clf = RandomForestClassifier(class_weight={0:1, 1:1998/1278},random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)
xgb_clf = XGBClassifier(random_state=42)
svc_cls = SVC(random_state=42)



In [ ]:
#voting regressor

voting_cls = VotingClassifier(
    estimators=[
        ('rf', rf_clf),#randomforest
        ('gb', gb_clf), # Gradientboosting
        ('xgb', xgb_clf), # Xgbclassifier

    ]
)

In [ ]:
#stacking
stacking_cls = StackingClassifier(
    estimators=[
     ('lgr', lgr),
    ('rf', rf_clf),
    ('gb', gb_clf),
    ('xgb', xgb_clf)
])

In [ ]:
model_train = {
    'Logistic Regression': lgr,
    'Decision Tree': dtc,
    'KNN': knn,
    'Random Forest': rf_clf,
    'Gradient Boosting': gb_clf,
    'XGBoost': xgb_clf,
    'SVC': svc_cls,
    'Voting Classifier': voting_cls,
    'Stacking Classifier': stacking_cls

}

In [ ]:
results = []
for name, model in model_train.items():
  pipe = Pipeline(
      [
          ('numercal_pipe',num_pipeline),
          ('model',model)
      ]
  )

  pipe.fit(X_train,y_train)

  y_pred = pipe.predict(X_test)

  accuracy = accuracy_score(y_test,y_pred)
  cls_rep = classification_report(y_test,y_pred)
  cnf_mat = confusion_matrix(y_test,y_pred)
  results.append(
      {
          'Model Name': name,
          'Accuracy': accuracy,
          'Classification Report': cls_rep,
          'Confusion Matrix': cnf_mat
      }
  )

  result_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
print(result_df)

In [ ]:
best_model = result_df.iloc[0]['Model Name']
best_model_obj = model_train[best_model]

final_pipe = Pipeline(
    [
        ('preprocessor',num_pipeline),
        ('model',best_model_obj)
    ]
)

final_pipe.fit(X_train,y_train)
y_final_pred = final_pipe.predict(X_test)
print("Final Accuracy:", accuracy_score(y_test, y_final_pred))

cm = confusion_matrix(y_test, y_final_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# cross validation
from sklearn.model_selection import cross_val_score
svc_pipe = Pipeline(
    [
        ('numercal_pipe',num_pipeline),
        ('model',SVC(class_weight='balanced',probability=True))
    ]
)

In [ ]:
svc_pipe.fit(X_train,y_train)

In [ ]:
y_svc_pred = svc_pipe.predict(X_test)

In [ ]:
acc = accuracy_score(y_test,y_svc_pred)
print(acc)
rep_cls = classification_report(y_test,y_svc_pred)
print(rep_cls)

In [ ]:
cross_val = cross_val_score(svc_pipe,X_train,y_train,cv=5,scoring='accuracy')
print(cross_val)

In [ ]:
print(np.mean(cross_val))

In [ ]:
print(np.std(cross_val))

In [ ]:
vtn_pipe = Pipeline(
    [
        ('numercal_pipe',num_pipeline),
        ('model',voting_cls)
    ]
)

In [ ]:
cross_val_vtn = cross_val_score(vtn_pipe,X_train,y_train,cv=5,scoring='accuracy')
print(cross_val_vtn)

In [ ]:
print(cross_val_vtn.mean())

In [ ]:
print(cross_val_vtn.std())

In [ ]:
param_grid = {
              'model__C': [0.01,0.1,1,10,100],
              'model__max_iter': [1000,5000]
              }

In [ ]:
rnd = RandomForestClassifier(class_weight={0:1, 1:1998/1278},random_state=42)
rnd.fit(X_train,y_train)
y_rnd_pred = rnd.predict(X_test)
print("Random :",accuracy_score(y_test,y_rnd_pred))
print(classification_report(y_test,y_rnd_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(
    estimator=svc_pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train,y_train)

In [ ]:
print("Best Parameters Found:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

In [ ]:
best_lr_model = grid_search.best_estimator_

In [ ]:
y_pred = best_lr_model.predict(X_test)
print("Final Test Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64,32),
    activation='relu',
    solver='adam',
    alpha=0.01,
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42
)

mlp_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('model', mlp)
])


mlp_pipe.fit(X_train, y_train)
y_pred = mlp_pipe.predict(X_test)

print(" MLPClassifier_Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
# Install CatBoost
!pip install catboost

from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    class_weights=[1,2],
    verbose=0
)

cat_pipe = Pipeline([
    ('num_pipe', num_pipeline),
    ('model', cat)
])

cat_pipe.fit(X_train, y_train)
y_pred_cat = cat_pipe.predict(X_test)

print(accuracy_score(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))

In [ ]:
print("Catboot", accuracy_score(y_test, y_pred_cat))
print(classification_report(y_test, y_pred_cat))

In [ ]:
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cat = CatBoostClassifier(
    iterations=800,
    depth=8,
    learning_rate=0.03,
    loss_function='Logloss',
    eval_metric='F1',
    class_weights=[1, 1.8],
    random_state=42,
    verbose=0
)

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', cat)
])

cat_pipe.fit(X_train, y_train)
y_pred = cat_pipe.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
# Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Load data
df = pd.read_csv("/content/water_potability.csv")

# Fill missing values for selected columns
for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df[col].fillna(df[col].median())

# Features & target
X = df.drop('Potability', axis=1)
y = df['Potability']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Numeric pipeline
# This num_pipeline will be unrolled and its steps used directly in ImbPipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())  # scaling for SVC / NN
])

# Apply SMOTE to handle imbalance
smote = SMOTE(random_state=42)

# Base models with class_weight
rf_clf = RandomForestClassifier(class_weight={0:1, 1:1.56}, random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)
xgb_clf = XGBClassifier(random_state=42, eval_metric='logloss')
svc_cls = SVC(class_weight={0:1, 1:1.56}, probability=True, random_state=42)

# Voting Classifier
voting_cls = VotingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf), ('xgb', xgb_clf)],
    voting='soft'
)

# Stacking Classifier
stacking_cls = StackingClassifier(
    estimators=[('rf', rf_clf), ('gb', gb_clf), ('xgb', xgb_clf)],
    final_estimator=LogisticRegression(class_weight={0:1, 1:1.56}, max_iter=1000)
)

# Model dictionary
model_train = {
    'Random Forest': rf_clf,
    'Gradient Boosting': gb_clf,
    'XGBoost': xgb_clf,
    'SVC': svc_cls,
    'Voting Classifier': voting_cls,
    'Stacking Classifier': stacking_cls
}

# Train and evaluate
results = []

for name, model in model_train.items():
    # Build pipeline with SMOTE + numeric pipeline steps integrated directly
    pipe = ImbPipeline([
        ('smote', smote),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
        ('model', model)
    ])

    # Fit model
    pipe.fit(X_train, y_train)

    # Predict probabilities & apply threshold
    if hasattr(pipe.named_steps['model'], 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)[:,1]
        threshold = 0.4  # tuned threshold for minority class
        y_pred = (y_prob >= threshold).astype(int)
    else:
        y_pred = pipe.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall1 = recall_score(y_test, y_pred)
    cls_report = classification_report(y_test, y_pred)
    cnf_mat = confusion_matrix(y_test, y_pred)

    results.append({
        'Model Name': name,
        'Accuracy': acc,
        'F1 (Class1)': f1,
        'Recall (Class1)': recall1,
        'Classification Report': cls_report,
        'Confusion Matrix': cnf_mat
    })

# Create results dataframe
result_df = pd.DataFrame(results).sort_values('F1 (Class1)', ascending=False)
print(result_df[['Model Name', 'Accuracy', 'F1 (Class1)', 'Recall (Class1)']])